# 04e. Veto de mãe

Censo sem CPF no [`04_atribuir.ipynb`](04_atribuir.ipynb) cujo **melhor par**
ainda falha o veto de mãe da escada atual.

A porta espelha o 04: iguais ou prefixo; JW ≥ 0,75 a partir de 0,50; mãe
contida (≥ 0,50); um nome (≥ 0,90); data nula (≥ 0,35 + JW ≥ 0,75 ou contida);
baixa nota (> 0,05 e < 0,50 + data nula/diferente |Δano| ≤ 10 + (JW ≥ 0,75 e mãe fora do top 10, ou contida com ≥ 3 tokens)); ou nome completo e data
iguais. Primeiro nome ausente ou com Levenshtein > 1 vai para `so_primeiro`.


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

splink_input → censo_limpo_aplicacao ∪ cpf_limpo_aplicacao


('censo_limpo_aplicacao', 'cpf_limpo_aplicacao')

In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

print(
    'Pares no parquet:',
    f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}",
)
print(
    'Atribuições:',
    f"{con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]:,}",
)


Pares no parquet: 6,598,211
Atribuições: 4,092,388


Melhor par de quem não saiu no 04. `veto_mae` exige primeiro
nome preenchido e com no máximo 1 de Levenshtein: se o primeiro nome já
reprova, o par vai para `so_primeiro` e não entra na leitura da mãe.
A porta da mãe segue as mesmas exceções do 04.


In [3]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_mae = f"least(len({tok_c}), len({tok_p}))"
prefixo_mae = (
    f"{n_mae} >= 2 AND "
    f"list_slice({tok_c}, 1, {n_mae}) = list_slice({tok_p}, 1, {n_mae})"
)
mae_jw = 'jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.75'
mae_igual_ou_prefixo = (
    f'ca.nome_mae_phon = pb.nome_mae_phon OR ({prefixo_mae})'
)
mae_contem = (
    f"{n_mae} >= 2 AND "
    f"len({tok_c}) <> len({tok_p}) AND "
    f"len(list_intersect({tok_c}, {tok_p})) = {n_mae} AND "
    f"NOT ({prefixo_mae})"
)
mae_contem_3 = (
    f"{n_mae} >= 3 AND "
    f"len({tok_c}) <> len({tok_p}) AND "
    f"len(list_intersect({tok_c}, {tok_p})) = {n_mae} AND "
    f"NOT ({prefixo_mae})"
)
mae_um_nome = (
    f"{n_mae} = 1 AND "
    f"len({tok_c}) <> len({tok_p}) AND "
    f"len(list_intersect({tok_c}, {tok_p})) = 1"
)
data_nula = 'ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL'
data_nula_ou_diferente = (
    f"({data_nula}) OR ("
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento <> pb.data_nascimento "
    "AND abs(year(CAST(ca.data_nascimento AS DATE)) - "
    "year(CAST(pb.data_nascimento AS DATE))) <= 10)"
)
mae_nao_frequente = (
    "ca.primeiro_nome_mae_phon NOT IN ("
    "SELECT primeiro_nome_mae_phon FROM top10_primeiro_mae)"
)
mae_data_nula = (
    f"p.match_probability >= 0.35 AND ({data_nula}) "
    f"AND (({mae_jw}) OR ({mae_contem}))"
)
mae_baixa_nota = (
    f"p.match_probability > 0.05 AND p.match_probability < 0.50 "
    f"AND ({data_nula_ou_diferente}) AND ("
    f"(({mae_jw}) AND ({mae_nao_frequente})) OR ({mae_contem_3})"
    f")"
)
mae_passa = (
    f"({mae_igual_ou_prefixo}) "
    f"OR (p.match_probability >= 0.50 AND ({mae_jw})) "
    f"OR (p.match_probability >= 0.50 AND ({mae_contem})) "
    f"OR (p.match_probability >= 0.90 AND ({mae_um_nome})) "
    f"OR ({mae_data_nula}) "
    f"OR ({mae_baixa_nota})"
)
excecao = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL "
    "AND ca.nome_completo_phon = pb.nome_completo_phon "
    "AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)

con.execute(f'''
CREATE OR REPLACE TABLE top10_primeiro_mae AS
SELECT primeiro_nome_mae_phon
FROM {SPLINK_INPUT_VIEW}
WHERE origem = 'censo'
  AND primeiro_nome_mae_phon IS NOT NULL
  AND primeiro_nome_mae_phon <> ''
GROUP BY 1
ORDER BY COUNT(*) DESC, primeiro_nome_mae_phon
LIMIT 10
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
      SELECT 1 FROM atribuicao_2 a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE melhor_par AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_sem AS
SELECT
    s.unique_id AS unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN 'sem_par'
        WHEN ca.primeiro_nome_phon IS NULL
             OR pb.primeiro_nome_phon IS NULL
             OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) > 1
            THEN 'so_primeiro'
        WHEN ca.nome_mae_phon IS NOT NULL
             AND pb.nome_mae_phon IS NOT NULL
             AND NOT ({mae_passa})
             AND NOT ({excecao})
            THEN 'veto_mae'
        ELSE 'passaria'
    END AS porta
FROM censo_sem_cpf s
LEFT JOIN melhor_par p ON p.unique_id_censo = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''')

display(con.execute('''
SELECT
    porta,
    COUNT(*) AS n,
    SUM(CASE WHEN match_probability >= 0.75 THEN 1 ELSE 0 END) AS n_p075,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM melhor_sem
GROUP BY 1
ORDER BY n DESC
''').df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,porta,n,n_p075,avg_p
0,sem_par,1463222,0.0,NaN
1,passaria,431255,175853.0,0.526
2,veto_mae,75829,15784.0,0.375
3,so_primeiro,16336,12522.0,0.754


No veto de mãe, como os dois nomes da mãe se afastam, ao lado
do nome da pessoa e da data. `mesmas_palavras_ordem` e `contem_sem_prefixo`
são o lugar em que a comparação pode estar rígida demais: as palavras batem,
a regra de prefixo não.


In [4]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_c = f"len({tok_c})"
n_p = f"len({tok_p})"
inter = f"len(list_intersect({tok_c}, {tok_p}))"

con.execute(f'''
CREATE OR REPLACE TABLE veto_mae AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.nome_mae_phon AS mae_censo,
    pb.nome_mae_phon AS mae_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) AS jw_mae,
    CASE
        WHEN list_sort({tok_c}) = list_sort({tok_p}) AND {tok_c} <> {tok_p}
            THEN 'mesmas_palavras_ordem'
        WHEN {tok_c} = {tok_p}
            THEN 'mesmas_palavras'
        WHEN {inter} = least({n_c}, {n_p}) AND {n_c} <> {n_p}
            THEN 'contem_sem_prefixo'
        WHEN jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.92
            THEN 'parecidas'
        WHEN {inter} >= 1
            THEN 'alguma_palavra'
        ELSE 'sem_palavra'
    END AS mae_dif,
    CASE
        WHEN ca.nome_completo_phon = pb.nome_completo_phon THEN 'exato'
        WHEN ca.primeiro_nome_phon = pb.primeiro_nome_phon
             AND ca.ultimo_nome_phon = pb.ultimo_nome_phon THEN 'pontas'
        ELSE 'outro'
    END AS nome_pessoa,
    CASE
        WHEN ca.data_nascimento IS NULL AND pb.data_nascimento IS NULL THEN 'as_duas_nulas'
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'uma_nula'
        WHEN ca.data_nascimento = pb.data_nascimento THEN 'exata'
        ELSE 'diferente'
    END AS data_par
FROM melhor_sem m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE m.porta = 'veto_mae'
''')

n_excecao = con.execute('''
SELECT COUNT(*) FROM veto_mae
WHERE nome_pessoa = 'exato' AND data_par = 'exata'
''').fetchone()[0]
print('Veto de mãe:', f"{con.execute('SELECT COUNT(*) FROM veto_mae').fetchone()[0]:,}")
print('Nome e data exatos dentro do veto (tem de ser 0):', n_excecao)

display(con.execute('''
SELECT
    mae_dif,
    nome_pessoa,
    data_par,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(jw_mae), 3) AS avg_jw_mae
FROM veto_mae
GROUP BY 1, 2, 3
ORDER BY n DESC
''').df())


Veto de mãe: 75,829
Nome e data exatos dentro do veto (tem de ser 0): 0


,mae_dif,nome_pessoa,data_par,n,pct,avg_p,avg_jw_mae
0,alguma_palavra,exato,diferente,26272,34.6,0.324,0.673
1,alguma_palavra,exato,uma_nula,12492,16.5,0.342,0.676
2,sem_palavra,exato,diferente,9012,11.9,0.382,0.593
3,sem_palavra,exato,uma_nula,6045,8.0,0.400,0.589
4,alguma_palavra,outro,exata,2774,3.7,0.790,0.664
5,alguma_palavra,pontas,exata,2524,3.3,0.630,0.737
6,sem_palavra,outro,exata,2490,3.3,0.791,0.592
7,alguma_palavra,pontas,diferente,1812,2.4,0.195,0.709
8,sem_palavra,pontas,exata,1640,2.2,0.703,0.609
9,sem_palavra,pontas,diferente,1631,2.2,0.154,0.605


Amostra. Primeiro as mães com as mesmas palavras ou uma dentro
da outra. Depois as parecidas com nome da pessoa exato. Por último, mãe sem
palavra em comum e nome da pessoa exato: o veto no caso claro.


In [5]:
cols = '''
    match_probability,
    mae_dif,
    nome_pessoa,
    data_par,
    ROUND(jw_mae, 3) AS jw_mae,
    nome_censo,
    nome_cpf,
    mae_censo,
    mae_cpf,
    dob_censo,
    dob_cpf
'''

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif IN ('mesmas_palavras_ordem', 'mesmas_palavras', 'contem_sem_prefixo')
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'parecidas' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'sem_palavra' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 20
''').df())


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.843428,contem_sem_prefixo,exato,diferente,0.656,KLARISI KARDOZU,KLARISI KARDOZU,EUNICE,MARIA EUNICE CARDOSO,2011-11-27,2011-09-27
1,0.795914,contem_sem_prefixo,outro,exata,0.655,MARIA LIVRAMENTU,MARIA LIVRAMENTU FONSEKA SOUZA,SOCORRO,MARIA SOCORRO SANTOS FONSECA,1981-09-29,1981-09-29
2,0.793191,contem_sem_prefixo,outro,exata,0.655,RAIMUNDU SANTUS,RAIMUNDU SANTUS FONSEKA,SOCORRO,MARIA SOCORRO SANTOS FONSECA,1992-08-17,1992-08-17
3,0.789888,contem_sem_prefixo,outro,exata,0.626,MARIA KLARA,MARIA KLARA SIUVA GONKAUVIS,FRANCISCA,MARIA FRANCISCA SOARES SILVA,2011-10-02,2011-10-02
4,0.782795,contem_sem_prefixo,pontas,exata,0.640,GUILERMI SOUZA XIMENDIS,GUILERMI MENDIS SOUZA XIMENDIS,CRISTINA,MARIA CRISTINA MENDES SOUZA,2016-01-07,2016-01-07
5,0.766668,contem_sem_prefixo,outro,exata,0.625,ANTONI KAUI,ANTONI KAUI SIUVA KRUS,LETICIA,MARIA LETICIA CRUZ REIS,2021-04-14,2021-04-14
6,0.499282,contem_sem_prefixo,pontas,diferente,0.693,JAIUSON GARSIAS SANTUS,JAIUSON GARSIA SANTUS,MARIA GARCIA,MARIA LIVRAMENTO SILVA GARCIA,2000-10-30,2002-10-30
7,0.499282,contem_sem_prefixo,pontas,diferente,0.878,JOZI UELINTON MOREIRA SANTANA,JOZI UELITON MOREIRA SANTANA,MARIA MOREIRA,MARIA ANTONIA MOREIRA,1998-02-12,1991-02-12
8,0.499282,mesmas_palavras_ordem,outro,diferente,0.956,LEONARDU SOUZA MORAU,LEONARDU SOUZA MOURAU,MARIA FATIMA SOUSA CARVALHO,MARIA FATIMA CARVALHO SOUSA,2002-06-21,2000-06-21
9,0.499282,contem_sem_prefixo,outro,diferente,0.915,SIUVESTRI FREITA,SIUVESTRI FREITAS,RAIMUNDA FREITAS,RAIMUNDA RITA FREITAS,1995-12-31,1992-12-31


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.499652,parecidas,exato,diferente,0.959,JOAU VITOR KONSEIKAU,JOAU VITOR KONSEIKAU,MARIA LELDE CARVALHO CONCEICAO,MARIA LEUDE CARVALHO CONCEICAO,2016-07-16,2017-06-16
1,0.498070,parecidas,exato,diferente,0.939,ANTONIU MONTEIRU SOUZA,ANTONIU MONTEIRU SOUZA,MARIA AMPARA MONTEIRO SOUSA,MARIA AMPARO MONTEIRO SOUSA,2001-09-19,2001-08-19
2,0.497607,parecidas,exato,diferente,0.937,LUIS NASIMENTU,LUIS NASIMENTU,MARIA MARGARETE NASCIMENTO,MARIA MARCHARETE NASCIMENTO,1980-08-25,1981-08-25
3,0.497310,parecidas,exato,diferente,0.950,MARIA JEZUS SIUVA AUMEIDA,MARIA JEZUS SIUVA AUMEIDA,MARIA APARECIDA DILVA ALMEIDA,MARIA APARECIDA SILVA ALMEIDA,2001-09-26,2001-09-25
4,0.497310,parecidas,exato,diferente,0.985,ELOIZA SOUZA PEREIRA,ELOIZA SOUZA PEREIRA,MARIA CONCEICAO ROCHA SOUSA,MARIA CONCEICAO ROCHA SOUZA,2013-11-10,2013-11-09
5,0.497310,parecidas,exato,diferente,0.993,FRANSISKU XAVIS SIUVA,FRANSISKU XAVIS SIUVA,MARIA SIPLICIO CHAVES SILVA,MARIA SIMPLICIO CHAVES SILVA,1969-05-23,1969-05-25
6,0.494576,parecidas,exato,diferente,0.992,FLAVIU SIUVA SOUZA,FLAVIU SIUVA SOUZA,MARIA LURDES SILVA SOUSA,MARIA LOURDES SILVA SOUSA,1977-09-27,1976-09-27
7,0.491583,parecidas,exato,diferente,0.953,MARIANA SOUZA SANTUS,MARIANA SOUZA SANTUS,MARIA LOURDES SOUSA SANTOS,MARIA LOURDES SOUZA SANTOS,2018-02-18,2017-02-18
8,0.491583,parecidas,exato,diferente,0.975,FELIPI LIMA SIUVA,FELIPI LIMA SIUVA,MARIA MAGNA LIMA,MARIA MAGDA LIMA,2006-12-28,2005-12-28
9,0.491583,parecidas,exato,diferente,0.953,MARIA APARESIDA FEREIRA SIUVA,MARIA APARESIDA FEREIRA SIUVA,MARIA LEILIANE COSTA SILVA,MARIA LEILIANE ROCHA SILVA,2013-02-23,2014-02-23


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.984569,sem_palavra,exato,diferente,0.683,JOZI MORAIS KARNEIRU,JOZI MORAIS KARNEIRU,MARIA JOSE PEREIRA,ELIETH MORAIS CARNEIRO,1980-08-26,1990-08-26
1,0.984569,sem_palavra,exato,diferente,0.587,MARIA ELAINI SOUZA MORAIS,MARIA ELAINI SOUZA MORAIS,ANALIA DUARTE FRANCA,ADELAIDE RODRIGUES SOUSA MORAIS,2001-08-29,2008-08-29
2,0.984569,sem_palavra,exato,diferente,0.601,MARIA EMANUELI MENDIS AUVIS,MARIA EMANUELI MENDIS AUVIS,JOSILEIA MORENA,LUZILEA MORENO MENDES,2007-05-31,2017-05-31
3,0.984569,sem_palavra,exato,diferente,0.653,RAIMUNDU GUSTAVU SANTUS SOUZA,RAIMUNDU GUSTAVU SANTUS SOUZA,ANA ELIZA,GIORLANA SANTOS,2017-04-13,2014-04-13
4,0.984569,sem_palavra,exato,diferente,0.508,JOZI EDIVAN ROXA GOMIS,JOZI EDIVAN ROXA GOMIS,RAIMUNDA PEREIRA SANTOS,JOANA ROCHA GOMES,1971-08-30,1975-08-30
5,0.984569,sem_palavra,exato,diferente,0.672,FRANSISKU PEDRU SIUVA AUVIS,FRANSISKU PEDRU SIUVA AUVIS,MARIA VIRGENS LOPES SOUSA,PATRICIA GOMES SILVA,2005-10-31,2008-10-31
6,0.984569,sem_palavra,exato,diferente,0.652,ANA VITORIA VIEIRA KONSEIKAU,ANA VITORIA VIEIRA KONSEIKAU,JULIANA CONCEICAO MARTINS OLIVEIRA,LEIDIANE VIEIRA SILVA,2021-08-30,2012-08-30
7,0.984569,sem_palavra,exato,diferente,0.593,MARIA ELENA LINDOZU GAUVAU,MARIA ELENA LINDOZU GAUVAU,SONIA MARIA SILVA RIBEIRO,FELOMENA LINDOSO GALVAO,1984-05-25,1982-05-25
8,0.984569,sem_palavra,exato,diferente,0.656,RAIMUNDU AKINU PINHEIRU JUNIOR,RAIMUNDU AKINU PINHEIRU JUNIOR,LUCIMEIRE BARROS PINHEIRO,LUCILENE CONCEICAO MORAES,2021-12-10,2001-12-10
9,0.984569,sem_palavra,exato,diferente,0.497,MARIA LUZIA LARANJEIRA SANTUS,MARIA LUZIA LARANJEIRA SANTUS,GERTRUDIZ AMORM,MARIA SOCORRO LARANJEIRA,1998-08-15,1988-08-15


In [6]:
con.close()
